In [1]:
import torch
import torch.nn.functional as F

# import numpy as np
import matplotlib.pyplot as plt  # for making figures

%matplotlib inline
# !pip install jupyter_black
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [2]:
# download the names.txt file from github
# !wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

In [3]:
# read in all the words
words = open("data/names.txt", "r").read().splitlines()
words[:8], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'],
 32033)

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [5]:
# build the dataset
block_size = (
    3  # context length: how many characters do we take to predict the next one?
)
n_embd = 10
n_hidden = 200
seed = 2147483647

# rng = np.random.default_rng(seed=seed)
# C = rng.standard_normal((vocab_size, n_embd))
C = (
    torch.randn(
        vocab_size,
        n_embd,
        generator=torch.Generator().manual_seed(seed),
        dtype=torch.float64,
    )
    * 0.01
)

C.shape, C[0]

(torch.Size([27, 10]),
 tensor([-0.0092, -0.0082, -0.0086, -0.0035,  0.0034, -0.0127,  0.0146,  0.0012,
         -0.0116, -0.0174], dtype=torch.float64))

In [6]:
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(C[context])
            Y.append(ix)
            context = context[1:] + [ix]  # crop and append

    X = torch.stack(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y


import random

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])  # 80%

Xdev, Ydev = build_dataset(words[n1:n2])  # 10%
Xte, Yte = build_dataset(words[n2:])  # 10%

torch.Size([182625, 3, 10]) torch.Size([182625])
torch.Size([22655, 3, 10]) torch.Size([22655])
torch.Size([22866, 3, 10]) torch.Size([22866])


In [7]:
from rnn import *

In [8]:
vocab_size, n_embd, n_hidden, block_size

(27, 10, 200, 3)

In [33]:
epochs = 100
batch_size = 32
lr = 0.01

model = RNN1D(
    input_dim=n_embd,
    hidden_dim=n_hidden,
    batch_size=batch_size,
    output_dim=vocab_size,
    time=block_size,
    generator=torch.Generator().manual_seed(seed),
)

model.Wxh[0][:2]

tensor([-0.0092, -0.0082], dtype=torch.float64, grad_fn=<SliceBackward0>)

In [34]:
def close(a, b) -> bool:
    with torch.no_grad():
        a = torch.from_numpy(a)
        b = b.detach()
        assert a.shape == tuple(b.shape)
        assert torch.allclose(a, b, rtol=1e-06)
    return True

In [35]:
for epoch in range(epochs):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    X_b, y_b = Xtr[ix], Ytr[ix]
    
    for p in [
        model.Wxh,
        model.Whh,
        model.Wyh,
        model.bh,
        model.by,
    ]:
        if p.grad is not None:
            p.grad = None

    y_t = model.forward(X_b)
    # close(y_t, y_torch)
    # print(X_b.shape)
    dy_t, dWyh, dh_t, dby, dWhh, dWxh, dbh = model.backward_step(X_b, y_t, lr)
    y_t.sum().backward()

    print(dWxh.shape,  model.Wxh.shape)
    assert torch.allclose(dWxh, model.Wxh.grad)
    assert torch.allclose(dWhh, model.Whh.grad)
    assert torch.allclose(dWyh, model.Wyh.grad)
    assert torch.allclose(dbh, model.bh.grad)
    assert torch.allclose(dby, model.by.grad)

    for p in [
        model.Wxh,
        model.Whh,
        model.Wyh,
        model.bh,
        model.by,
    ]:
        p.data -= lr * p.grad

    # print(f'loss = {}')

RuntimeError: output with shape [1, 27] doesn't match the broadcast shape [32, 27]